In [1]:
import brightway2 as bw
from brightway2 import *
import time
import numpy as np
import openpyxl
import os
import pandas as pd
import glob

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
bw.projects.set_current('Prospective LCA v4') # set current project

In [4]:
databaseNames = bw.databases
myDatabaseNames = []
for databaseName in databaseNames:
    if 'Abhi' in databaseName and 'image' in databaseName:
        myDatabaseNames.append(databaseName)

In [5]:
myDatabaseNames

['lci-Abhi image SSP2-Base 2020',
 'lci-Abhi image SSP2-Base 2030',
 'lci-Abhi image SSP2-Base 2040',
 'lci-Abhi image SSP2-Base 2050',
 'lci-Abhi image SSP2-RCP19 2030',
 'lci-Abhi image SSP2-RCP19 2040',
 'lci-Abhi image SSP2-RCP19 2050',
 'lci-Abhi image SSP2-RCP26 2030',
 'lci-Abhi image SSP2-RCP26 2040',
 'lci-Abhi image SSP2-RCP26 2050']

In [22]:
methodsDict = {'climate change GWP 20a': ('IPCC 2013', 'climate change', 'GWP 20a, incl. H and bio CO2')}    

In [23]:
methodsList = []
hydrogenResultsFileNames = []
carbonDioxideResultsFileNames = []
ammoniaResultsFileNames = []
methanolResultsFileNames = []
ethyleneResultsFileNames = []
for keys, values in methodsDict.items():  
    methodsList.append(values)
    hydrogenResultsFileNames.append('hydrogen ' + keys + ' results.xlsx')
    carbonDioxideResultsFileNames.append('carbon dioxide ' + keys + ' results.xlsx')
    ammoniaResultsFileNames.append('ammonia ' + keys + ' results.xlsx')
    methanolResultsFileNames.append('methanol ' + keys + ' results.xlsx')
    ethyleneResultsFileNames.append('ethylene ' + keys + ' results.xlsx')

allResultsFileNames = hydrogenResultsFileNames + carbonDioxideResultsFileNames + ammoniaResultsFileNames + methanolResultsFileNames + ethyleneResultsFileNames

In [24]:
def lca_calculations(activities, methodsList):
    results = np.zeros((len(activities), len(methodsList)))
    lca = LCA({activities[0] : 1}, method = methodsList[0]) # LCA object which will do all calculating
    lca.lci() # calculate inventory once to load all database data
    lca.decompose_technosphere() # keep the LU factorized matrices for faster calculations
    lca.lcia() # load method data
    characterizationMatrices = []
    
    for method in methodsList:
        lca.switch_method(method)
        characterizationMatrices.append(lca.characterization_matrix.copy())
        
    for first, activity in enumerate(activities):
        lca.redo_lci({activity : 1})
        
        for second, matrix in enumerate(characterizationMatrices):
            results[first, second] = (matrix * lca.inventory).sum()
            
    return results

In [25]:
def check_or_create_excel_file(filePath):
    if not os.path.exists(filePath):
        wb = openpyxl.Workbook()
        wb.save(filePath)
        print(f'Excel file created at {filePath}')

In [26]:
def delete_first_sheet_from_excel_files(filePath):
    wb = openpyxl.load_workbook(filePath)
    if wb.sheetnames[0] == 'Sheet':  # check if the workbook has any sheets
        firstSheet = wb.sheetnames[0]  # get the name of the first sheet
        wb.remove(wb[firstSheet])  # remove the first sheet
        wb.save(filePath)

In [27]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'hydrogen, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(hydrogenResultsFileNames)):
        hydrogenResultsFilePath = os.path.join('..', 'Results', 'GWP 20a', hydrogenResultsFileNames[i])
        check_or_create_excel_file(hydrogenResultsFilePath)
        with pd.ExcelWriter(hydrogenResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\GWP 20a\hydrogen climate change GWP 20a results.xlsx


In [28]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'carbon dioxide, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(carbonDioxideResultsFileNames)):
        carbonDioxideResultsFilePath = os.path.join('..', 'Results', 'GWP 20a', carbonDioxideResultsFileNames[i])
        check_or_create_excel_file(carbonDioxideResultsFilePath)
        with pd.ExcelWriter(carbonDioxideResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\GWP 20a\carbon dioxide climate change GWP 20a results.xlsx


In [29]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'ammonia, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(ammoniaResultsFileNames)):
        ammoniaResultsFilePath = os.path.join('..', 'Results', 'GWP 20a', ammoniaResultsFileNames[i])
        check_or_create_excel_file(ammoniaResultsFilePath)
        with pd.ExcelWriter(ammoniaResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\GWP 20a\ammonia climate change GWP 20a results.xlsx


In [30]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'methanol, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(methanolResultsFileNames)):
        methanolResultsFilePath = os.path.join('..', 'Results', 'GWP 20a', methanolResultsFileNames[i])
        check_or_create_excel_file(methanolResultsFilePath)
        with pd.ExcelWriter(methanolResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\GWP 20a\methanol climate change GWP 20a results.xlsx


In [31]:
for myDatabaseName in myDatabaseNames:
    allInventories = []
    allNames = []
    allLocations = []
    myDatabase = bw.Database(myDatabaseName)
    for activity in myDatabase:
        if 'ethylene, Abhi' in activity['reference product']:
            allInventories.append(activity)
            allNames.append(activity['name'])
            allLocations.append(activity['location'])
    lcaScores = lca_calculations(allInventories, methodsList)
    for i in range(len(ethyleneResultsFileNames)):
        ethyleneResultsFilePath = os.path.join('..', 'Results', 'GWP 20a', ethyleneResultsFileNames[i])
        check_or_create_excel_file(ethyleneResultsFilePath)
        with pd.ExcelWriter(ethyleneResultsFilePath, engine = 'openpyxl', mode = 'a') as writer:
            lcaScoresSpecific = lcaScores[:, i].ravel().tolist()
            lcaScoresDF = pd.DataFrame({'Activity' : allNames, 'Location' : allLocations, 'Value' : lcaScoresSpecific})
            lcaScoresDFPivot = lcaScoresDF.pivot(index = 'Activity', columns = 'Location', values = 'Value')
            if myDatabaseName in writer.book.sheetnames:
                writer.book.remove(writer.book[myDatabaseName])
            lcaScoresDFPivot.to_excel(writer, sheet_name = myDatabaseName.replace("lci-Abhi ", ""))

Excel file created at ..\Results\GWP 20a\ethylene climate change GWP 20a results.xlsx


In [32]:
allResultsFileNames = glob.glob(os.path.join('..', 'Results', 'GWP 20a', '*.xls*'), recursive = True)
for filePath in allResultsFileNames:
    delete_first_sheet_from_excel_files(filePath)